# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/vedant08mehta/Flyrank-assignment1.git
%cd /content/Flyrank-assignment1

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 141 (delta 49), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.88 MiB | 10.59 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/Flyrank-assignment1


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I will use page-level performance and content features that are available at prediction time: impressions_90d, clicks_90d, ctr, avg_position, engagement_rate, content_age_days, days_since_last_update, word_count, search_volume, competition, and content_type. Numerical features will be kept as numeric, while content_type will be encoded as a categorical feature. Missing numerical values will be handled with median imputation and missing categorical values will be treated as a separate category.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "search_volume",
    "competition",
    "content_type"
]

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "search_volume",
    "competition"
]

categorical_features = [
    "content_type"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X = df[feature_cols]

X_ready = preprocessor.fit_transform(X)

print(f"Original feature columns: {len(feature_cols)}")
print(f"Encoded feature matrix shape: {X_ready.shape}")

Original feature columns: 11
Encoded feature matrix shape: (30000, 13)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d and clicks_90d measure recent search exposure and clicks. ctr measures clicks relative to impressions, while avg_position represents average search position. engagement_rate describes the proportion of sessions that were engaged. content_age_days, days_since_last_update, and word_count describe the content itself. search_volume and competition describe the search environment. content_type is categorical and will be one-hot encoded. Numerical missing values are median-imputed and the categorical feature uses the most frequent observed category. These features are treated as available at the prediction point, subject to the leakage checks below.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Feature availability and missing values:\n")

for col in feature_cols:
    print(
        f"{col:25} "
        f"missing={df[col].isna().sum():,}"
    )

Feature availability and missing values:

impressions_90d           missing=0
clicks_90d                missing=0
ctr                       missing=0
avg_position              missing=0
engagement_rate           missing=0
content_age_days          missing=0
days_since_last_update    missing=0
word_count                missing=7,699
search_volume             missing=2,468
competition               missing=2,468
content_type              missing=0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I will check three main leakage risks: columns derived directly from the target, variables that use information from after the prediction point, and fields that may encode the outcome indirectly. In particular, trend_direction and trend_pct are suspicious because the target is derived from trend_direction. I will also check whether any selected feature is actually a future-window measurement rather than information available when the prediction is made.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target = (
    df["trend_direction"] == "down"
).astype(int)

print("=== Target distribution ===")
print(target.value_counts())
print()


print("=== Potential label-derived fields ===")

suspicious = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

for col in suspicious:
    print(
        f"{col:20} "
        f"present={col in df.columns}"
    )


print("\n=== Feature overlap check ===")

for col in feature_cols:
    print(
        f"{col:25} "
        f"target_correlation={df[col].corr(target):.3f}"
        if pd.api.types.is_numeric_dtype(df[col])
        else f"{col:25} categorical"
    )


print("\n=== Explicit exclusions ===")

excluded = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print(excluded)

=== Target distribution ===
trend_direction
1    16262
0    13738
Name: count, dtype: int64

=== Potential label-derived fields ===
trend_direction      present=True
trend_pct            present=True
is_declining_label   present=False

=== Feature overlap check ===
impressions_90d           target_correlation=-0.018
clicks_90d                target_correlation=-0.040
ctr                       target_correlation=-0.062
avg_position              target_correlation=-0.029
engagement_rate           target_correlation=-0.013
content_age_days          target_correlation=-0.164
days_since_last_update    target_correlation=0.081
word_count                target_correlation=0.090
search_volume             target_correlation=-0.019
competition               target_correlation=-0.009
content_type              categorical

=== Explicit exclusions ===
['trend_direction', 'trend_pct', 'is_declining_label']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

trend_direction — excluded because the target is defined directly from whether this field is "down", so using it would give the model the answer.

trend_pct — excluded because it is another direct description of the performance trend used to define the outcome and would create target leakage.

is_declining_label — excluded because it is the target itself and cannot be used as an input feature.

Any feature calculated from a future period relative to the prediction timestamp would also be excluded, because it would not have been available when the prediction was made.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

leakage_in_features = [
    col
    for col in excluded_features
    if col in feature_cols
]

print("Excluded fields:")
for col in excluded_features:
    print(f"- {col}")

print(
    "\nLeakage fields accidentally included "
    f"in feature_cols: {leakage_in_features}"
)

assert len(leakage_in_features) == 0

print("\nLeakage check passed: no explicitly excluded "
      "label-derived fields are in the feature vector.")

Excluded fields:
- trend_direction
- trend_pct
- is_declining_label

Leakage fields accidentally included in feature_cols: []

Leakage check passed: no explicitly excluded label-derived fields are in the feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.